In [4]:
import torch
import torch.nn as nn

class WelfordLayerNorm(nn.Module):

    def __init__(self, features: int, eps: float = 1e-6) -> None:
        super().__init__()
        self.eps = eps
        self.alpha = nn.Parameter(torch.ones(features)) # alpha is a learnable parameter
        self.bias = nn.Parameter(torch.zeros(features)) # bias is a learnable parameter
        self.features = features

    def forward(self, x):
        # x: (batch, seq_len, hidden_size)
        batch_size, seq_len, _ = x.shape
        mean = torch.zeros(batch_size, seq_len, self.features, device=x.device)
        M2 = torch.zeros(batch_size, seq_len, self.features, device=x.device)
        
        for i in range(seq_len):
            xi = x[:, i, :]
            delta = xi - mean[:, i, :]
            mean[:, i, :] += delta / (i + 1)
            M2[:, i, :] += delta * (xi - mean[:, i, :])
        population_var = M2 / seq_len
        std = torch.sqrt(population_var + self.eps)
        return self.alpha * (x - mean) / std + self.bias
    
# Test the WelfordLayerNorm class
if __name__ == "__main__":  
    welford_layer_norm = WelfordLayerNorm(features=4)
    x = torch.randn(2, 3, 4)  # Example input tensor
    output = welford_layer_norm(x)
    print("Input:\n", x)
    print("Output:\n", output)
    print("Alpha:\n", welford_layer_norm.alpha)
    print("Bias:\n", welford_layer_norm.bias)
    print("Output shape:", output.shape)  # Should be the same shape as input
    print("Mean of output:", output.mean(dim=-1))  # Should be close to zero
    print("Std of output:", output.std(dim=-1))  # Should be close to one
    print("Epsilon:", welford_layer_norm.eps)  # Should be the same as defined
    print("Alpha shape:", welford_layer_norm.alpha.shape)  # Should be (4,)
    print("Bias shape:", welford_layer_norm.bias.shape)  # Should be (4,)
    print("WelfordLayerNorm parameters:", list(welford_layer_norm.parameters()))  # Should show alpha and bias
    print("WelfordLayerNorm state_dict:", welford_layer_norm.state_dict())  # Should show alpha and bias

Input:
 tensor([[[-0.8430,  0.1424, -0.1826, -1.6285],
         [-0.2780, -1.4292, -0.8328,  0.2079],
         [-0.9455, -1.6537,  0.8820, -0.0638]],

        [[-0.4663, -1.2257,  0.4716,  0.2129],
         [-1.0538,  0.1978, -0.8530,  0.0715],
         [ 1.1636,  1.2626,  0.3333, -0.0255]]])
Output:
 tensor([[[ 0.0000,  0.0000,  0.0000,  0.0000],
         [-1.2247, -1.2247, -1.2247,  1.2247],
         [-1.4142, -1.4142,  1.4142, -1.4134]],

        [[ 0.0000,  0.0000,  0.0000,  0.0000],
         [-1.2247,  1.2247, -1.2247,  1.2240],
         [ 1.4142,  1.4142,  1.4142, -1.4094]]], grad_fn=<AddBackward0>)
Alpha:
 Parameter containing:
tensor([1., 1., 1., 1.], requires_grad=True)
Bias:
 Parameter containing:
tensor([0., 0., 0., 0.], requires_grad=True)
Output shape: torch.Size([2, 3, 4])
Mean of output: tensor([[ 0.0000e+00, -6.1238e-01, -7.0691e-01],
        [ 0.0000e+00, -2.0081e-04,  7.0831e-01]], grad_fn=<MeanBackward1>)
Std of output: tensor([[0.0000, 1.2247, 1.4141],
        [0.00

In [ ]:
import torch
import torch.nn as nn

class WelfordLayerNorm(nn.Module):

    def __init__(self, features: int, eps: float = 1e-6) -> None:
        super().__init__()
        self.eps = eps
        self.alpha = nn.Parameter(torch.ones(features)) # alpha is a learnable parameter
        self.bias = nn.Parameter(torch.zeros(features)) # bias is a learnable parameter
        self.features = features

    def forward(self, x):
        # x: (batch, seq_len, hidden_size)
        batch_size, seq_len, _ = x.shape
        mean = torch.zeros(batch_size, seq_len, self.features, device=x.device)
        M2 = torch.zeros_like(mean)

        for i in range(seq_len):
            xi = x[:, i]
            delta = xi - mean
            mean += delta / (i + 1)
            delta2 = xi - mean
            m2 += delta * delta2imimimp


In [21]:
import torch

def welford_online(x: torch.Tensor) -> torch.Tensor:
    """
    x: 1D tensor of values [x1, x2, x3, ..., xn]
    Returns: mean, variance, std
    """
    n = 0
    mean = torch.tensor(0.0, device=x.device)
    M2 = torch.tensor(0.0, device=x.device)

    for xi in x:
        n += 1
        delta = xi - mean
        mean += delta / n
        delta2 = xi - mean
        M2 += delta * delta2

    if n < 2:
        return float('nan')
    else:
        variance = M2 / n # or use m2 / (n - 1) for unbiased variance
        return mean, variance
    
#===== Example ===========
x = torch.randn(100)
mean, variance = welford_online(x)
print("Welford Mean:", mean.item())
print("Welford Variance:", variance.item())
print("Torch Reference:", x.mean().item(), x.var(unbiased=False).item())

Welford Mean: -0.09614933282136917
Welford Variance: 1.036554217338562
Torch Reference: -0.09614931046962738 1.0365544557571411
